In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="evaluation",
    notebook_name="01_classification_metrics.ipynb"
)

# Classification Metrics: Accuracy, Precision, Recall, F1 & Confusion Matrix

## What You'll Learn

In this notebook, we'll learn the **most fundamental metrics** used to evaluate AI models that make predictions. By the end, you'll understand:

- **Accuracy** -- What percentage did the model get right?
- **Precision** -- When the model says "yes", how often is it correct?
- **Recall** -- Of all the actual "yes" cases, how many did the model find?
- **F1 Score** -- A balanced combination of precision and recall
- **Confusion Matrix** -- A table showing all correct and incorrect predictions

No prior ML knowledge needed! We'll use simple, real-world examples.

---
## 1. Setup

We only need two libraries. `sklearn` (scikit-learn) is the most popular ML library for metrics.
`matplotlib` helps us draw pretty pictures.

In [ ]:
# Install if needed (uncomment the line below)
# !pip install scikit-learn matplotlib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
import matplotlib.pyplot as plt

print("All set! Let's learn about metrics.")

---
## 2. Our Scenario: The Spam Filter

Imagine you built an AI spam filter for your email. It looks at each email and
predicts: **"spam"** or **"not spam"**.

Let's say you received 10 emails. Here's what ACTUALLY happened vs what your
AI PREDICTED:

```
Email 1: Actually SPAM    -> AI said: SPAM      (correct!)
Email 2: Actually SPAM    -> AI said: SPAM      (correct!)
Email 3: Actually SPAM    -> AI said: NOT SPAM   (oops -- missed it!)
Email 4: Actually SPAM    -> AI said: SPAM      (correct!)
Email 5: Actually NOT SPAM -> AI said: NOT SPAM   (correct!)
Email 6: Actually NOT SPAM -> AI said: SPAM      (oops -- false alarm!)
Email 7: Actually NOT SPAM -> AI said: NOT SPAM   (correct!)
Email 8: Actually NOT SPAM -> AI said: NOT SPAM   (correct!)
Email 9: Actually NOT SPAM -> AI said: NOT SPAM   (correct!)
Email 10: Actually NOT SPAM -> AI said: NOT SPAM  (correct!)
```

Let's put this into code:

In [ ]:
# What actually happened (the truth)
# 1 = spam, 0 = not spam
actual =    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

# What the AI predicted
predicted = [1, 1, 0, 1, 0, 1, 0, 0, 0, 0]

# Let's see them side by side
print("Email | Actual    | Predicted | Correct?")
print("-" * 45)
for i in range(len(actual)):
    label_actual = "SPAM" if actual[i] == 1 else "NOT SPAM"
    label_predicted = "SPAM" if predicted[i] == 1 else "NOT SPAM"
    correct = "Yes" if actual[i] == predicted[i] else "NO"
    print(f"  {i+1:2d}  | {label_actual:9s} | {label_predicted:9s} | {correct}")

---
## 3. Accuracy -- The Simplest Metric

**Accuracy** answers: *"Out of ALL emails, what percentage did the AI get right?"*

```
Accuracy = Number correct / Total number
```

Simple!

In [ ]:
# Calculate accuracy by hand
correct_count = sum(1 for a, p in zip(actual, predicted) if a == p)
total = len(actual)
accuracy_manual = correct_count / total

print(f"Correct predictions: {correct_count} out of {total}")
print(f"Accuracy (by hand):  {accuracy_manual:.0%}")
print()

# Now using sklearn (same answer, less work!)
accuracy = accuracy_score(actual, predicted)
print(f"Accuracy (sklearn):  {accuracy:.0%}")

### Why Accuracy Alone Isn't Enough

Accuracy can be **misleading** when your data is imbalanced (one class is much
more common than the other).

**Example:** Imagine 1000 emails, but only 10 are spam. A "dumb" model that
ALWAYS says "not spam" would be 99% accurate! But it catches ZERO spam.

Let's see this in action:

In [ ]:
# Imbalanced example: 990 real emails, 10 spam
actual_imbalanced = [0] * 990 + [1] * 10  # 990 not spam, 10 spam
dumb_model =        [0] * 1000             # Always says "not spam"

dumb_accuracy = accuracy_score(actual_imbalanced, dumb_model)
print(f"The 'always say not spam' model has {dumb_accuracy:.1%} accuracy!")
print(f"But it catches {sum(1 for a, p in zip(actual_imbalanced, dumb_model) if a == 1 and p == 1)} out of 10 spam emails.")
print()
print("This is why we need more metrics than just accuracy!")

---
## 4. Precision -- "When I say spam, am I right?"

**Precision** answers: *"Of all the emails the AI FLAGGED as spam, how many actually WERE spam?"*

```
                   True Positives
Precision = ──────────────────────────────
            True Positives + False Positives
```

In plain English: **"How trustworthy is the spam label?"**

- High precision = When it says spam, it's almost always right (few false alarms)
- Low precision = It cries "spam!" too often, catching real emails too

In [ ]:
# Let's calculate precision by hand for our spam filter
#
# The AI said "spam" for emails: 1, 2, 4, 6
# Of those, emails 1, 2, 4 were ACTUALLY spam (correct!)
# Email 6 was NOT spam (false alarm!)

true_positives = sum(1 for a, p in zip(actual, predicted) if a == 1 and p == 1)
false_positives = sum(1 for a, p in zip(actual, predicted) if a == 0 and p == 1)

precision_manual = true_positives / (true_positives + false_positives)

print(f"True Positives (correctly caught spam):  {true_positives}")
print(f"False Positives (false alarms):          {false_positives}")
print(f"Precision (by hand): {precision_manual:.0%}")
print()

# Using sklearn
precision = precision_score(actual, predicted)
print(f"Precision (sklearn): {precision:.0%}")
print()
print("Meaning: When our AI says 'spam', it's right 75% of the time.")

---
## 5. Recall -- "Did I find ALL the spam?"

**Recall** answers: *"Of all the emails that WERE spam, how many did the AI catch?"*

```
              True Positives
Recall = ──────────────────────────────
         True Positives + False Negatives
```

In plain English: **"How thorough is the spam filter?"**

- High recall = Catches almost all spam (doesn't let much through)
- Low recall = Misses a lot of spam (they slip into your inbox)

In [ ]:
# Let's calculate recall by hand
#
# There were 4 spam emails total: 1, 2, 3, 4
# The AI caught 3 of them: 1, 2, 4
# It MISSED email 3 (false negative)

false_negatives = sum(1 for a, p in zip(actual, predicted) if a == 1 and p == 0)

recall_manual = true_positives / (true_positives + false_negatives)

print(f"True Positives (correctly caught spam):  {true_positives}")
print(f"False Negatives (missed spam):           {false_negatives}")
print(f"Recall (by hand): {recall_manual:.0%}")
print()

# Using sklearn
recall = recall_score(actual, predicted)
print(f"Recall (sklearn): {recall:.0%}")
print()
print("Meaning: Our AI catches 75% of all spam emails (misses 25%).")

---
## 6. The Precision-Recall Tradeoff

Here's the key insight: **Precision and Recall pull in opposite directions!**

```
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  More aggressive (flag more as spam):                       │
│    Recall goes UP   (catches more real spam)                │
│    Precision goes DOWN (more false alarms)                  │
│                                                             │
│  More conservative (only flag obvious spam):                │
│    Precision goes UP (fewer false alarms)                   │
│    Recall goes DOWN  (misses more spam)                     │
│                                                             │
│  You can't maximize both at the same time!                  │
└─────────────────────────────────────────────────────────────┘
```

Let's demonstrate this:

In [ ]:
# Same actual labels
actual_demo = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

# Conservative model: only flags when VERY sure -> high precision, low recall
conservative = [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]  # Only flags 1 email

# Aggressive model: flags everything suspicious -> low precision, high recall
aggressive =   [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]  # Flags 7 emails

print("Conservative model (only flags when sure):")
print(f"  Precision: {precision_score(actual_demo, conservative):.0%} (when it says spam, it's always right!)")
print(f"  Recall:    {recall_score(actual_demo, conservative):.0%} (but it only catches 1 out of 4 spam)")
print()

print("Aggressive model (flags everything suspicious):")
print(f"  Precision: {precision_score(actual_demo, aggressive):.2%} (lots of false alarms)")
print(f"  Recall:    {recall_score(actual_demo, aggressive):.0%} (but it catches ALL the spam!)")

---
## 7. F1 Score -- The Best of Both Worlds

Since precision and recall trade off, we often want ONE number that balances both.
That's the **F1 score** -- the harmonic mean of precision and recall.

```
         2 x Precision x Recall
F1 = ─────────────────────────────
         Precision + Recall
```

Why "harmonic mean" instead of a regular average? Because the harmonic mean
**punishes extreme imbalances**. If either precision OR recall is very low,
F1 will be low too.

In [ ]:
# F1 for our original spam filter
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)

print(f"Precision: {precision_manual:.0%}")
print(f"Recall:    {recall_manual:.0%}")
print(f"F1 (by hand): {f1_manual:.0%}")
print()

# Using sklearn
f1 = f1_score(actual, predicted)
print(f"F1 (sklearn): {f1:.0%}")
print()

# Now let's see why F1 is better than a regular average
print("--- Why F1 punishes imbalance ---")
print()
print("Model with Precision=100%, Recall=10%:")
regular_avg = (1.0 + 0.1) / 2
f1_imbalanced = 2 * (1.0 * 0.1) / (1.0 + 0.1)
print(f"  Regular average: {regular_avg:.0%} (looks okay!)")
print(f"  F1 score:        {f1_imbalanced:.0%} (reveals the problem!)")

---
## 8. The Confusion Matrix -- See Everything at Once

A **confusion matrix** is a 2x2 table that shows ALL the predictions organized
by what happened:

```
                    Predicted: NOT SPAM    Predicted: SPAM
                   ┌────────────────────┬────────────────────┐
Actually NOT SPAM  │  True Negative (TN)│ False Positive (FP)│
                   │  "Correct! Not spam"│ "Oops! False alarm"│
                   ├────────────────────┼────────────────────┤
Actually SPAM      │ False Negative (FN)│  True Positive (TP)│
                   │ "Oops! Missed spam"│ "Correct! Got spam"│
                   └────────────────────┴────────────────────┘
```

Let's create one:

In [ ]:
# Calculate the confusion matrix
cm = confusion_matrix(actual, predicted)

print("Confusion Matrix (raw numbers):")
print(cm)
print()
print(f"True Negatives  (correctly said 'not spam'): {cm[0][0]}")
print(f"False Positives (wrongly said 'spam'):       {cm[0][1]}")
print(f"False Negatives (missed real spam):          {cm[1][0]}")
print(f"True Positives  (correctly caught spam):     {cm[1][1]}")

In [ ]:
# Now let's make a pretty visual version!
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Spam", "Spam"]
)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Spam Filter Confusion Matrix", fontsize=14, fontweight="bold")
ax.set_xlabel("What the AI Predicted", fontsize=12)
ax.set_ylabel("What Actually Happened", fontsize=12)
plt.tight_layout()
plt.show()

print("\nReading the matrix:")
print("- Top-left (5): correctly identified as NOT spam")
print("- Top-right (1): false alarm -- real email marked as spam")
print("- Bottom-left (1): missed spam -- it got through")
print("- Bottom-right (3): correctly caught spam")

---
## 9. The Classification Report -- All Metrics in One Shot

sklearn has a handy function that shows ALL the metrics at once.
This is what you'll use in practice most often:

In [ ]:
# The classification report gives you everything!
report = classification_report(
    actual,
    predicted,
    target_names=["Not Spam", "Spam"]
)

print("Full Classification Report:")
print("=" * 55)
print(report)
print("How to read this:")
print("- Each row is a class (Not Spam, Spam)")
print("- Precision/Recall/F1 are shown per class")
print("- 'support' = how many of each class exist in the data")
print("- 'macro avg' = simple average across classes")
print("- 'weighted avg' = average weighted by number of samples")

---
## 10. Practice: Disease Detection Scenario

Let's try a more realistic (and higher-stakes) example: **disease detection**.

A hospital AI looks at blood test results and predicts:
"Does this patient have the disease?" (1 = yes, 0 = no)

In medical tests:
- **False Positive** = Telling a healthy person they're sick (scary but fixable with more tests)
- **False Negative** = Telling a sick person they're healthy (**dangerous!** They might not get treatment)

So for disease detection, **Recall is more important than Precision!**
We'd rather have some false alarms than miss sick patients.

In [ ]:
# Disease detection scenario
# 20 patients: 5 have the disease, 15 are healthy
actual_disease = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
predicted_disease = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1]

print("Disease Detection Results")
print("=" * 40)
print(f"Accuracy:  {accuracy_score(actual_disease, predicted_disease):.0%}")
print(f"Precision: {precision_score(actual_disease, predicted_disease):.0%}")
print(f"Recall:    {recall_score(actual_disease, predicted_disease):.0%}")
print(f"F1 Score:  {f1_score(actual_disease, predicted_disease):.0%}")
print()

# Confusion matrix
cm_disease = confusion_matrix(actual_disease, predicted_disease)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_disease,
    display_labels=["Healthy", "Has Disease"]
)
disp.plot(ax=ax, cmap="Reds", values_format="d")
ax.set_title("Disease Detection Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nThe model missed {cm_disease[1][0]} patient(s) who actually had the disease!")
print("In medicine, this is the MOST dangerous type of error.")
print("That's why Recall matters so much here.")

---
## 11. Multi-Class: More Than Two Categories

So far we've looked at **binary** classification (spam/not spam, sick/healthy).
But what if the model classifies things into **3 or more categories**?

Example: Classifying animal photos as **cat**, **dog**, or **bird**.

In [ ]:
# Multi-class example: animal classification
actual_animals = ["cat", "cat", "cat", "dog", "dog", "dog", "bird", "bird", "bird", "bird"]
pred_animals =   ["cat", "cat", "dog", "dog", "dog", "bird", "bird", "bird", "bird", "cat"]

# Full report
print("Animal Classification Report:")
print("=" * 55)
print(classification_report(actual_animals, pred_animals))

# Visual confusion matrix
cm_animals = confusion_matrix(actual_animals, pred_animals, labels=["cat", "dog", "bird"])
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_animals,
    display_labels=["Cat", "Dog", "Bird"]
)
disp.plot(ax=ax, cmap="Greens", values_format="d")
ax.set_title("Animal Classifier Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nReading this matrix:")
print("- Diagonal (top-left to bottom-right) = correct predictions")
print("- Off-diagonal = mistakes")
print("- Look for patterns: which animals get confused for each other?")

---
## 12. Summary -- When to Use What

| Metric | Use When... | Example |
|--------|------------|--------|
| **Accuracy** | Classes are balanced and all errors are equal | "Is this photo a cat or dog?" |
| **Precision** | False alarms are costly | Search results (don't show irrelevant results) |
| **Recall** | Missing positive cases is costly | Disease detection (don't miss sick patients) |
| **F1 Score** | You need to balance precision and recall | Most real-world classification tasks |
| **Confusion Matrix** | You want to see the full picture | Debugging where your model goes wrong |

### Key Takeaways

1. **Accuracy alone is not enough** -- especially with imbalanced data
2. **Precision vs Recall is a tradeoff** -- you can't maximize both
3. **F1 Score balances both** -- use it as your default "single number"
4. **Context matters** -- in medicine, recall matters more; in search, precision matters more
5. **Always look at the confusion matrix** -- it shows you WHERE the model makes mistakes

---

## What You Just Built

You now understand the core classification metrics used across all of machine learning:
- **Accuracy** — the overall hit rate, and why it fails under class imbalance
- **Precision** — trustworthiness of positive predictions
- **Recall** — completeness of positive predictions
- **F1 Score** — the balanced grade that punishes extreme imbalances
- **Confusion Matrix** — the full picture of what your model gets right and wrong

For the full mathematical treatment (micro/macro/weighted averaging, ROC-AUC, PR-AUC, calibration, Brier score) and staff-level interview questions, see the [interview deep-dive](./classification-metrics-interview.md).

In [ ]:
# Quick reference: calculate all metrics at once
print("Quick Reference: All metrics for our spam filter")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(actual, predicted):.2%}")
print(f"Precision: {precision_score(actual, predicted):.2%}")
print(f"Recall:    {recall_score(actual, predicted):.2%}")
print(f"F1 Score:  {f1_score(actual, predicted):.2%}")
print()
print("Confusion Matrix:")
print(confusion_matrix(actual, predicted))
print()
print("Done! You now understand the most important classification metrics.")

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)